# **Construct a Multimodal Vector Index**


## Introduction

Retrieval-Augmented Generation (RAG) systems depend critically on the quality of their **retrieval layer** to produce grounded and contextually accurate responses.做出有根据的和上下文准确的反应。

In real-world applications, user queries often span heterogeneous data sources, including structured textual information and visual content.在实际应用程序中，用户查询经常跨越异构数据源， 包括结构化文本信息和可视化内容。As a result, modern RAG pipelines increasingly adopt **multimodal retrieval architectures** capable of indexing and searching across multiple data modalities. 因此，现代RAG管道越来越多地 采用能够跨多种数据模式进行索引和搜索的多模式检索体系结构。

To support these scenarios, multimodal systems encode different data types into dense vector representations that enable efficient similarity-based search.为了支持这些场景，多模态系统将不同的数据类型编码为密集的向量表示，从而实现高效的基于相似性的搜索

A well-designed retrieval layer ensures that high-quality evidence is identified before any downstream reasoning or generation occurs.设计良好的检索层可确保在任何下游推理或生成发生之前识别出高质量的证据

In this lab, you will build the retrieval backbone of a multimodal RAG system for restaurant and food discovery.您将构建用于餐厅和食物发现的多式联运RAG系统的检索主干  Using structured outputs from Module 1 together with raw food images, you will construct **vectorized, searchable indexes** that support cross-modal evidence retrieval.使用模块1的结构化输出 和 原始食物图像，您将构建 支持 跨模式证据检索的 矢量化、可搜索索引

The lab emphasizes **retrieval pipeline engineering**, 该实验室强调“检索管道工程”， focusing on how heterogeneous data is transformed, embedded, and organized within scalable vector databases using ChromaDB. 重点研究如何使用ChromaDB在可扩展的矢量数据库中转换、嵌入和组织异构数据

This modular design mirrors production RAG systems, where the retrieval layer operates as a reusable component independent of the generation model.这种模块化设计反映了生产RAG系统，其中检索层作为独立于生成模型的可重用组件进行操作
Upon completion, you will have a functional multimodal vector index that forms the foundation for similarity search, metadata-constrained retrieval, and multimodal fusion in subsequent lessons.你将有一个功能的 多模态向量索引，形成基础的相似性搜索，元数据约束检索


## Objectives

After completing this lab, you will be able to:

- Construct retrieval-ready documents from structured restaurant data  从结构化的餐厅数据中构建可检索的文档
- Generate dense embeddings for both text and image modalities 生成密集的嵌入文本和图像模式 
- Build separate vector indexes for articles and food images using Chroma  使用色度 为文章和 食物图像 建立单独的矢量索引
- Persist multimodal vector databases for downstream retrieval tasks  为下游 检索 任务 持久化 多模态矢量数据库  
- Prepare the retrieval backbone for similarity search and metadata filtering  为相似性搜索 和元数据 过滤 准备检索主干

** These capabilities form the foundation of a production-style multimodal RAG retrieval pipeline这些功能构成了生产风格的多模态RAG检索管道的基础 **


## Set up the lab environment

In this step, you'll install the core libraries required to build a multimodal retrieval system.您将安装构建多模式检索系统所需的核心库

The environment supports both text and image processing and prepares the runtime for vector database construction. These components will be used throughout the module.支持文本和图像处理，并准备运行时的矢量数据库建设。这些组件将在整个模块中使用

The environment includes:

- **PyTorch (CPU)** for model inference and tensor operations  用于模型推理和张量操作 
- **LangChain + Chroma** for vector database construction and persistence  用于向量数据库的构建和持久化
- **Sentence-Transformers** for generating text embeddings  
用于生成文本嵌入的sentence - transforme
- **Transformers (CLIP)** for image embedding support  变压器（剪辑）**图像嵌入支持
- **Pillow and NumPy** for image loading and numerical processing  
Pillow和NumPy**用于图像加载和数值处理

After installation completes, the environment will be ready for multimodal vector index construction 安装完成后，环境将为多模态矢量索引构建做好准


In [ ]:
# ================================
# Install dependencies
# ================================
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu
!pip install -q langchain==0.3.27 langchain-community==0.3.31 langchain-chroma==0.2.6
!pip install -q sentence-transformers==2.7.0 transformers pillow numpy


## Import core libraries

Next, import the core Python libraries used throughout this lab.
接下来，导入整个实验中使用的核心Python库

These libraries enable:
- structured data loading and preprocessing 结构化数据的加载和预处理
- embedding model initialization  嵌入模型初始化
- multimodal document construction 多式联运单据构建 
- vector database creation and management 矢量数据库的创建和管理 
Successful import confirms that the runtime environment is correctly configured and ready for the retrieval pipeline.成功导入确认运行时环境已正确配置，并为检索管道做好了准备。


In [ ]:
# ================================
# Import libraries
# ================================

# Standard library
import glob
import json
import os
import shutil
from pathlib import Path

# Third-party library
import numpy as np
import torch
from PIL import Image
from langchain_chroma import Chroma
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPProcessor

print("✅ Environment ready")


## Prepare the image dataset

Multimodal retrieval systems require both textual and visual evidence.
多模式检索系统需要文字和视觉证据

In this step, you'll perform the following operations:在这一步中，您将执行以下操作
1. Download the synthetic recipe image dataset 
下载合成配方图像数据集
2. Extract the image archive  提取图像存档
3. Collect image file paths for embedding 收集图像文件路径用于嵌入 

These images will later form the **Restaurant & Food Images database**, which complements the restaurant text database built from structured data.
这些图像稍后将形成**餐馆和食物图像数据库**，它补充了从结构化数据构建的餐馆文本数据库。

In [ ]:
# ================================
# Download and prepare image data
# ================================
ZIP_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/5_Rr6ohviItzucyWk6nkrw/synthetic-recipe-images.zip"
ZIP_PATH = "synthetic-recipe-images.zip"
IMG_DIR  = "recipe_images"

!wget -q -O {ZIP_PATH} {ZIP_URL}
!unzip -oq {ZIP_PATH} -d {IMG_DIR}

image_paths = sorted(glob.glob(f"{IMG_DIR}/**/*.png", recursive=True))
print(f"✅ Images found: {len(image_paths)}")


## Load the structured data

This module builds directly on the structured outputs produced earlier. Before constructing the multimodal index, load the processed JSON files generated earlier.该模块直接构建在前面生成的结构化输出上。在构造多模式索引之前，加载前面生成的处理过的JSON文件

You'll load:

- **Structured restaurant data**, which provides the textual retrieval corpus  
**结构化餐厅数据**，提供文本检索语料
- **Augmented recipe data**, which supplies additional metadata for food items  
- **增强配方数据**，为食品提供额外的元数据

These structured records will be transformed into retrieval-ready documents in the next step. 在下一步中，这些结构化记录将被转换为可检索的文档

Maintaining this pipeline connection ensures the multimodal RAG system remains modular and extensible.
保持这种管道连接可确保多模式RAG系统保持模块化和可扩展性


In [ ]:
# ================================
# Load structured data from Module 1
# ================================

# NOTE: Ensure the JSON files are in your current working directory.确保JSON文件在你当前的工作目录中。

with open("structured_restaurant_data.json", "r") as f:
    restaurants = json.load(f)

with open("augmented_food_recipe.json", "r") as f:
    recipes = json.load(f)

print(f"✅ Loaded restaurants: {len(restaurants)}")
print(f"✅ Loaded recipes:     {len(recipes)}")


## Initialize embedding models

To support multimodal retrieval, you need to initialize two specialized embedding models.
为了支持多模态检索，您需要初始化两个专门的嵌入模型

The system uses:

- **Sentence-Transformers (384-dimensional)** for restaurant text encoding  
句子变形金刚（384维）餐厅文本编码
- **CLIP image encoder (512-dimensional)** for food image encoding  
用于食品图像编码

Both embedding outputs are L2-normalized to support cosine similarity search in later retrieval stages. 
Using modality-appropriate encoders ensures that semantic similarity is preserved within each vector space.使用适合模态的编码器可确保在每个向量空间内保持语义相似性


In [ ]:
# ================================
# Initialize embedding models
# ================================

# ---- Text embedding model (384-d) ----
text_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_texts(texts, batch_size=64):
    return text_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        normalize_embeddings=True,  # cosine-ready
    ).astype(np.float32)

print("✅ Text embedder ready")


# ---- Image embedding model (512-d) ----
device = "cpu"
clip_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_name).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_name, use_fast=True)
clip_model.eval()

@torch.no_grad()
def embed_images(paths, batch_size=16):
    vecs = []
    for i in range(0, len(paths), batch_size):
        batch = paths[i:i+batch_size]
        imgs = [Image.open(p).convert("RGB") for p in batch]
        inputs = clip_processor(images=imgs, return_tensors="pt").to(device)
        feats = clip_model.get_image_features(**inputs)          # (B,512)
        feats = feats / feats.norm(dim=-1, keepdim=True)         # cosine-ready
        vecs.append(feats.cpu().numpy().astype(np.float32))
    return np.vstack(vecs)

print("✅ Image embedder ready")


## Construct multimodal documents

Vector databases operate on structured documents rather than raw files.

In this step, you'll construct two document sets:

1. **Article documents** derived from structured restaurant data  
来源于结构化的餐厅数据
2. **Image documents** derived from recipe image files  
图像文档来源于配方图像文件
Each document includes:
- Retrieval text (`page_content`)  
- A unique document identifier  
- Metadata fields that enable filtering and constrained search  
每份文件包括：
-检索文本（' page_content '）
—唯一的文档标识符
—启用过滤和约束搜索的元数据字段
These multimodal documents form the foundation of the retrieval index and will be embedded in the next stage.
这些多模式文档构成了检索索引的基础，并将在下一阶段进行嵌入。

In [ ]:
# ================================
# Build article and image documents
# ================================

# -------- articles --------
article_docs = []

for i, r in enumerate(restaurants):
    name = str(r.get("name", "")).strip()
    if not name:
        continue

    text = (
    f"Restaurant: {name}\n"
    f"Cuisine: {r.get('food_style','')}\n"
    f"Location: {r.get('location','')}"
    )

    # GUARANTEED UNIQUE
    doc_id = f"rest_{i}"

    article_docs.append(
        Document(
            page_content=text.strip(),
            metadata={
                "doc_id": doc_id,
                "cuisine": r.get("food_style"),
                "location": r.get("location"),
                "source": "restaurant",
            },
        )
    )

print("✅ article docs:", len(article_docs))


# -------- images --------
image_docs = []

for i, (p, rec) in enumerate(zip(image_paths, recipes)):
    doc_id = f"img_{i}"

    image_docs.append(
        Document(
            # keeps retrieval results readable
            page_content=rec.get("name", f"recipe image {i}"),
            metadata={
                "doc_id": doc_id,
                "image_path": p,
                "source": "recipe_image",
                "recipe_id": rec.get("id"),
                "cuisine": rec.get("cuisine"),
            },
        )
    )

print("✅ image docs:", len(image_docs))


## Construct the vector index 
You'll now construct the retrieval backbone of the multimodal RAG system.
您将构建多模式RAG系统的检索主干
This step performs the following operations:
1. Generate embeddings for both text and image documents  
为文本和图像文档生成嵌入
2. Create separate Chroma collections for each modality  
为每个模态创建单独的色度集合
3. Store vectors together with their metadata  
**将向量与其元数据一起存储**
4. Persist the multimodal index to disk for reuse  
**将多模态索引持久化到磁盘以供重用**


After this step completes, the system is fully prepared for similarity retrieval and metadata filtering in Lesson 2.


In [ ]:
# ================================
# Construct and Persist Multimodal Vector Indexes 构建和持久化多模态向量索引
# ================================

DB_DIR = str((Path.home() / "chroma_multimodal").resolve())

if os.path.isdir(DB_DIR):
    shutil.rmtree(DB_DIR)  # Reset vector DB (important for reruns)

# ----- article DB -----
A = embed_texts([d.page_content for d in article_docs])

article_db = Chroma(
    collection_name="restaurant_articles",
    persist_directory=DB_DIR,
)

article_db._collection.upsert(
    ids=[d.metadata["doc_id"] for d in article_docs],
    embeddings=A.tolist(),
    documents=[d.page_content for d in article_docs],
    metadatas=[d.metadata for d in article_docs],
)

print("✅ Article DB ready")


# ----- image DB -----
V = embed_images([d.metadata["image_path"] for d in image_docs])

image_db = Chroma(
    collection_name="food_images",
    persist_directory=DB_DIR,
)

image_db._collection.upsert(
    ids=[d.metadata["doc_id"] for d in image_docs],
    embeddings=V.tolist(),
    documents=[d.page_content for d in image_docs],
    metadatas=[d.metadata for d in image_docs],
)

print("✅ Image DB ready")
print("🎉 Multimodal Vector Index Construction COMPLETE")


### 📸 Screenshot Submission Requirement

At the end of this lab, take a screenshot of the final code cell and its output, and name it **M2L1_multimodal_vector_index.jpg**.

Your screenshot must clearly show:
- Successful creation of both Chroma collections:
  - `restaurant_articles`
  - `food_images`
- The final completion message:  
  **"Multimodal Vector Index Construction COMPLETE"**

This screenshot will later serve as a component for the assessment.


## Conclusion

In this lab, you built the multimodal retrieval foundation for the restaurant RAG system. You converted structured restaurant data and food images into retrieval-ready documents, generated embeddings, and constructed persistent vector indexes using Chroma.

The multimodal index is now prepared for similarity search and metadata filtering. 


## Authors


[Zikai Dou](https://author.skills.network/instructors/zikai_dou)


Copyright © IBM Corporation. All rights reserved.
